# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and available record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their corresponding `@id` values.

Let's inspect the record sets and a summary of their fields/columns. All references are by their `@id`.

In [ ]:
# Get all record sets in the dataset. Each record set contains its own fields/columns.
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets were found for this dataset. Please check the schema definition or contact the data provider.")
else:
    print("Available Record Sets:")
    for rs in record_sets:
        print(f"- RecordSet Name: {rs.name}, @id: {rs.id}")
        print("  Fields (by @id):")
        for field in rs.fields:
            print(f"    * {field.name} (@id: {field.id})")
        print("")

# Save record set @ids for later extraction
record_set_ids = [rs.id for rs in record_sets]

## 3. Data Extraction
Load data from each record set into pandas DataFrames for analysis. We use the record set and field `@id`s found above. If you are exploring a particular record set, set `record_set_to_explore` accordingly.

In [ ]:
dataframes = {}
for record_set_id in record_set_ids:
    # Extract records for each record set using its @id
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from record set: {record_set_id}")

# Select the first record set for demonstration purposes
if record_set_ids:
    record_set_to_explore = record_set_ids[0]
    df = dataframes[record_set_to_explore]
    print(f"\nColumns in record set '{record_set_to_explore}':\n{df.columns.tolist()}")
    display(df.head())
else:
    print('No available record sets to load data from.')

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic data processing steps:

- Filter records by a numeric field
- Normalize a numeric field
- Group by a categorical field (if present)

**Note:** Adjust `numeric_field_id` and `group_field_id` to reference actual field `@id` values present in the loaded DataFrame.

In [ ]:
# Example field IDs (please confirm these with the actual field list in your dataset)
# You can change these values to other @id's found using the overview steps.
numeric_field_id = None
group_field_id = None
for col in df.columns:
    # Automatic detection (for demo, look for typical numeric or group column names)
    if numeric_field_id is None and ('log_likelihood' in col.lower() or 'coef' in col.lower() or 'value' in col.lower()):
        numeric_field_id = col
    if group_field_id is None and ('ward' in col.lower() or 'county' in col.lower() or 'group' in col.lower()):
        group_field_id = col

if numeric_field_id is None:
    # Fallback: pick the first float/integer column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field:   {group_field_id if group_field_id else '[none found]'}")

if numeric_field_id:
    # Remove NaNs for clean EDA
    filtered_df = df[df[numeric_field_id].notnull()]
    threshold = filtered_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]) else 0
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group analysis
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset. Here are a few basic plotting examples:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Boxplot grouped by group field, if available
if group_field_id and group_field_id in df.columns and numeric_field_id:
    plt.figure(figsize=(12,6))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset via its Croissant schema, inspected available record sets and fields via `@id`, and performed exploratory analysis and visualization. Data organization is discoverable and addressable through Croissant `@id`s, supporting reproducible data science workflows with `mlcroissant`.

Further analysis can be performed by exploring other record sets, numeric and categorical fields using their unique `@id` references, or by transforming and modeling the extracted tabular data.